# Dataset Replay Lab

Purpose: replay captured or synthetic datasets through staged processors and compare outputs.

## Milestone 2 Packet Inspection

Peek at ESP32 packet bytes and use the future parser when it exists. The cell falls back to a canonical ADR-110 sync sample if no fixture is present.

In [ ]:
from pathlib import Path
import importlib

magic_names = {
    0xC5110001: "raw_csi",
    0xC511A110: "sync",
    0xC5110002: "edge_vitals",
    0xC5110003: "feature_vector",
    0xC5110004: "fused_vitals_or_wasm_output",
    0xC5110005: "compressed_csi",
}

fixture_candidates = [
    Path("tests/fixtures/esp32_raw_csi.bin"),
    Path("tests/fixtures/esp32_sync_packet.bin"),
    Path("tests/fixtures/esp32_packet.bin"),
    Path("../tests/fixtures/esp32_raw_csi.bin"),
    Path("../tests/fixtures/esp32_sync_packet.bin"),
    Path("../tests/fixtures/esp32_packet.bin"),
]

packet = None
for fixture in fixture_candidates:
    if fixture.exists():
        packet = fixture.read_bytes()
        print(f"Loaded {fixture} ({len(packet)} bytes)")
        break

if packet is None:
    packet = bytes.fromhex(
        "10a111c509010600"
        "f26db70100000000"
        "c5aca50100000000"
        "1400000000000000"
    )
    print("No fixture found; using canonical ADR-110 sync sample")

magic = int.from_bytes(packet[:4], "little") if len(packet) >= 4 else None
print({
    "length": len(packet),
    "magic": f"0x{magic:08x}" if magic is not None else None,
    "kind": magic_names.get(magic, "unknown"),
    "head_hex": packet[:32].hex(),
})

try:
    esp32 = importlib.import_module("ruview.protocols.esp32")
except ModuleNotFoundError as exc:
    esp32 = None
    print(f"Future parser unavailable yet ({exc.name}); header inspection still works")

if esp32 is not None:
    for name in ("parse_packet", "parse_esp32_packet", "parse_csi_frame", "parse_sync_packet"):
        parser = getattr(esp32, name, None)
        if callable(parser):
            try:
                parsed = parser(packet)
            except Exception as exc:
                print(f"{name} raised {type(exc).__name__}: {exc}")
            else:
                print(f"{name} -> {parsed!r}")
            break
    else:
        print("ruview.protocols.esp32 imported, but no known parse entry point was found")
